# Chat.DT — Building 15 Cypher run (Outlines + HF, GPU)

Runs the two Cypher settings (`CYPHER_SOFT`, `CYPHER_STRICT`) from a Colab GPU
runtime against a remote Chat.DT API, over the 5-question Ch9 demo set on the
DABC Building 15 model.

**Architecture**

- **Server (`api` container)** owns Neo4j (Bolt is internal-only), the test set,
  pre-executed gold queries, Cypher execution, and SVR/SCR/EA scoring.
- **This notebook** owns the LLM. It loads the server-built `bundle_<model>.json`
  to rehydrate vocabulary + IDS schema + model dump, runs Outlines-constrained
  decoding locally on the GPU, and POSTs each generated output to `/evaluate`.

Bolt never leaves the Docker network. The LLM never leaves Colab.

**Inputs**

1. `bundle_b15_ids.json`, built by `scripts/build_bundle.py` **after** the
   graph-stats merge (it must carry 8 relationship types — check for
   `BOUNDED_BY`; an older bundle has 4 and the two Hard queries become
   unanswerable).
2. The HTTPS `API_BASE_URL` of your Chat.DT deployment.
3. `CHATDT_API_TOKEN` in Colab Secrets, matching the server's `API_BEARER_TOKEN`.
4. An HF model id or local checkpoint (Outlines requires a local model for STRICT).

**Server-side prerequisite**

The `api` service must have `TEST_SET_PATH=/app/data/ch9_demo_b15.json`, otherwise
it falls back to the stale 28-case `data/test_set.json`. Cell 5 asserts on this.

Use an A100 / L4 runtime. Note the Outlines FSM compiles once at `setup()` and is
cached for the whole run — expect a one-off pause there, not per question.

## 1. Clone the repo and install GPU deps

In [1]:
%%bash
set -e
if [ ! -d Chat.DT ]; then
  git clone --depth 1 https://github.com/IvansSmirnoff/Chat.DT.git
fi
cd Chat.DT
pip install -q -r requirements-base.txt
# Full LLM stack: torch, transformers, outlines, etc.
pip install -q -r requirements-llm.txt
python -c 'import torch; print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name() if torch.cuda.is_available() else "no gpu")'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 313.9/313.9 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 MB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.3/102.3 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 51.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.8/67.8 kB 7.4 MB/s eta 0:00:00
cuda: True NVIDIA A100-SXM4-80GB


Cloning into 'Chat.DT'...


In [ ]:
import os, sys
sys.path.insert(0, 'Chat.DT')

# --- Remote Chat.DT API ---
os.environ['API_BASE_URL'] = 'https://chatdt.giovannidamilano.org'

# The bearer token is a SECRET and must never be committed. Store it in Colab
# Secrets (key icon in the left sidebar) as CHATDT_API_TOKEN; the getpass
# branch is the fallback for a plain Jupyter runtime.
try:
    from google.colab import userdata
    os.environ['API_BEARER_TOKEN'] = userdata.get('CHATDT_API_TOKEN')
except Exception:
    import getpass
    os.environ['API_BEARER_TOKEN'] = getpass.getpass('API_BEARER_TOKEN: ')

# --- LLM (runs in this Colab runtime) ---
os.environ['LLM_PROVIDER']   = 'local'
os.environ['LLM_MODEL_NAME'] = 'Qwen/Qwen2.5-14B-Instruct'   # HF repo id or /content/<path>

# Pydantic settings reads NEO4J_* with non-empty defaults; we never connect to
# Neo4j from Colab, but the import path still constructs Settings — leaving the
# defaults from src/config.py is fine.

## 2. Upload the bundle and ping the API

Drag `bundle_<model>.json` into the Colab file pane (or mount Drive). Then run a
readiness check against the API — bad token / unreachable Neo4j surfaces here
before any LLM work happens.

In [ ]:
from pathlib import Path
from src.client.api_client import ApiClient

# bundle_b15_ids.json is tracked, so the clone in cell 2 already has it —
# no manual upload. Falls back to the file pane if you dropped one in.
BUNDLE_PATH = Path('Chat.DT/data/bundle_b15_ids.json')
if not BUNDLE_PATH.exists():
    BUNDLE_PATH = Path('bundle_b15_ids.json')
assert BUNDLE_PATH.exists(), f'Bundle not found: {BUNDLE_PATH}'

import json
_b = json.load(open(BUNDLE_PATH))
_rels = set(_b['combined_vocabulary'].get('all_relations') or [])
print('bundle:', BUNDLE_PATH, '| relations:', sorted(_rels))
assert 'BOUNDED_BY' in _rels, (
    'Bundle predates the graph-stats merge (4 relationship types instead of 8). '
    'Rebuild with scripts/build_bundle.py — otherwise the two Hard queries are '
    'undecodable.'
)

client = ApiClient(os.environ['API_BASE_URL'], os.environ['API_BEARER_TOKEN'])
print('health:', client.health())
print('ready: ', client.health_ready())

# The server must be on Building 15, not Barcelona. IfcCovering/IfcShadingDevice
# are B15 markers; IfcColumn/IfcRoof/IfcWallStandardCase are Barcelona ones.
vocab = client.get_vocabulary() if hasattr(client, 'get_vocabulary') else None
if vocab:
    print('server entities:', len(vocab['entities']), '| relations:', vocab['relations'])
    assert 'BOUNDED_BY' in vocab['relations'], (
        'Server graph has no BOUNDED_BY — it is still the Barcelona model. '
        'Re-run the ETL on the server with building15_arc.ifc.'
    )

cases = client.get_test_set()
print('test cases on server:', len(cases))
for c in cases:
    print(' ', c.get('difficulty', '?'), '|', c['question'])

# The server falls back to data/test_set.json when TEST_SET_PATH is unset,
# which is the stale Barcelona 28-case set. Fail loudly rather than silently
# benchmarking the wrong questions.
assert len(cases) == 5, (
    f'Expected the 5-case Ch9 demo, got {len(cases)}. Set TEST_SET_PATH='
    '/app/data/ch9_demo_b15.json on the api service and restart it.'
)

## 3. Run the Cypher settings via the API

The runner loads the bundle locally, builds the LLM engine on this GPU, and
streams `(question, predicted_output)` to `POST /evaluate` for scoring.

`setup()` compiles the Outlines FSM **once** and caches it for the whole run.
Measured for this grammar against the Qwen2.5-14B-Instruct vocabulary
(151,652 tokens): **~74 s, 31,892 DFA states, ~870 MB** — on CPU, so expect the
same order in Colab. If it instead raises

```
Failed to build DFA number of DFA states exceeds limit of 2147483647
```

then `max_where_clauses` has drifted upward. That cap is the load-bearing one:
4 fails in every configuration measured (with predicates, without them, with a
narrow RETURN), while 1–2 succeed. Check that
`build_cypher_regex_from_vocabulary` still applies `max_where_clauses=1` and
`max_return_items=3`.

In [ ]:
from src.config import ExperimentSetting
from src.client.runner import ApiExperimentRunner, ApiRunnerConfig

# Single source of truth for the run name — the save cell globs on it. Hardcoding
# the prefix in two places is how the first run wrote empty result folders.
RUN_NAME = 'b15_ch9_demo'

# Cypher arm only. The Direct-QA settings are not run on Building 15: the model
# dump is 5,949 elements (~1.96 M tokens) against a 32 K context, so the
# dump-it-into-the-prompt approach is infeasible here rather than merely weak —
# that is the reportable finding, not an omission.
config = ApiRunnerConfig(
    bundle_path=BUNDLE_PATH,
    output_dir=Path('results'),
    name=RUN_NAME,
    settings=[
        ExperimentSetting.CYPHER_SOFT,    # prompt-only constraints
        ExperimentSetting.CYPHER_STRICT,  # + Outlines regex-constrained decoding
    ],
)

runner = ApiExperimentRunner(client=client, config=config)
runner.setup()

# Sanity checks before spending GPU time. Both must hold or the run is invalid.
eng = runner.engine
assert eng._regex_pattern, 'no constraint regex was built — CYPHER_STRICT would silently fall back to soft'
assert eng._strict_generator is not None, 'Outlines FSM failed to compile; see the warning above'
print('regex chars      :', len(eng._regex_pattern))
print('relationships    :', sorted(runner.vocabulary.all_relations))
print('chat template    :', bool(getattr(eng._tokenizer, 'chat_template', None)))

all_rows = runner.run_comparison()

In [ ]:
# Per-question view — this, not the means, is what Ch9 reports. n=5 is a
# demonstration on a real building, not a benchmark; no mean is meaningful here.
for setting, rows in all_rows.items():
    print('=' * 78)
    print(setting)
    print('=' * 78)
    for r in rows:
        ok = 'HIT ' if r['ea'] == 1.0 else 'MISS'
        print(f"[{r['index']}] {ok} {r['difficulty']:6s} {r['question']}")
        print(f"      gold: {r['gold_cypher']}")
        print(f"      pred: {r['generated_cypher'] or '(none)'}")
        print(f"      syntax_valid={r['is_valid_syntax']} ea={r['ea']} f1={r['f1']:.2f}"
              f" rows_pred={r['num_generated_results']} rows_gold={r['num_gold_results']}"
              + (f" error={r['error']}" if r['error'] else ''))
        print()

## 4. Inspect summaries (and persist to Drive if you like)

In [ ]:
import json, glob, os, shutil
from datetime import datetime

RUN_NAME = globals().get('RUN_NAME', 'b15_ch9_demo')

summaries = sorted(glob.glob(f'results/{RUN_NAME}_*_summary.json'))
assert summaries, (
    f'No summaries matching results/{RUN_NAME}_*_summary.json. '
    f'Present: {sorted(glob.glob("results/*"))}'
)

for path in summaries:
    print(path)
    print(json.dumps(json.load(open(path))['metrics'], indent=2))
    print()

date_str   = datetime.now().strftime('%Y-%m-%d_%H-%M')
model_name = os.environ.get('LLM_MODEL_NAME', 'unknown_model').replace('/', '_')

from google.colab import drive
drive.mount('/content/drive')
drive_base_dir = '/content/drive/MyDrive/chat_dt_results'
os.makedirs(drive_base_dir, exist_ok=True)

for path in summaries:
    # results/<RUN_NAME>_<setting>_summary.json -> <setting>
    setting = os.path.basename(path)[len(RUN_NAME) + 1:-len('_summary.json')]

    folder_name = f'saved_results_{date_str}_{model_name}_{RUN_NAME}_{setting}'
    local_dir = folder_name
    drive_dir = os.path.join(drive_base_dir, folder_name)
    os.makedirs(local_dir, exist_ok=True)
    os.makedirs(drive_dir, exist_ok=True)

    files = glob.glob(f'results/{RUN_NAME}_{setting}*')
    assert files, f'No result files for {setting} — refusing to create an empty folder'
    for file_path in files:
        shutil.copy(file_path, local_dir)
        shutil.copy(file_path, drive_dir)

    print(f'{setting}: copied {len(files)} files -> {drive_dir}/')